# Cycle 2 — Modelling (Chronological Split)

**Twin of `notebooks/cycle2_modelling.ipynb`**.

The processed Wyscout file (`wyscout_shots_processed.csv`) does not include any time/date column, so we re-load the raw `events_England.json`, attach the `matchId` (which is sequential by kickoff), and recompute the same nine features used in the original notebook. We then split on `matchId`: the last 20% of matches go to the test set.

In [ ]:
import sys, os

# Locate project root (folder containing data/, models/, notebooks/)
_here = os.getcwd()
while not os.path.isdir(os.path.join(_here, 'data')):
    _p = os.path.dirname(_here)
    if _p == _here: raise RuntimeError('project root not found')
    _here = _p
if _here not in sys.path:
    sys.path.insert(0, _here)

from config import Paths, ensure_dirs
ensure_dirs()  # creates models/cycle1-3 if missing


## Setup

In [3]:
import json, math, warnings
import pandas as pd
import numpy as np
warnings.filterwarnings('ignore')

from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, roc_auc_score,
                              classification_report, roc_curve)
from xgboost import XGBClassifier
import matplotlib.pyplot as plt

## Re-load raw events with matchId, build features

In [4]:
with open(str(Paths.EVENTS_ENGLAND)) as f:
    raw = json.load(f)

df = pd.DataFrame(raw)
shots = df[df['eventName'] == 'Shot'].copy().reset_index(drop=True)
print(f'Raw shots: {len(shots):,}')

# Goal target (tag 101)
shots['Goal'] = shots['tags'].apply(lambda tags: int(any(t['id']==101 for t in tags)))

# Tags 401/402/403 -> body part
shots['Left_Foot']  = shots['tags'].apply(lambda tags: int(any(t['id']==401 for t in tags)))
shots['Right_Foot'] = shots['tags'].apply(lambda tags: int(any(t['id']==402 for t in tags)))
shots['Header']     = shots['tags'].apply(lambda tags: int(any(t['id']==403 for t in tags)))

# Match half
shots = shots[shots['matchPeriod'].isin(['1H','2H'])].copy().reset_index(drop=True)
shots['First_Half'] = (shots['matchPeriod']=='1H').astype(int)

# Coordinates from positions[0]
shots['X'] = shots['positions'].apply(lambda p: p[0]['x'])
shots['Y'] = shots['positions'].apply(lambda p: p[0]['y'])

# Distance & angle (same constants as api/ml/pipeline.py)
PITCH_L, PITCH_W = 105.0, 68.0
POST_L, POST_R   = 30.34, 37.66
GOAL_Y = (POST_L+POST_R)/2
x_m = shots['X']/100.0 * PITCH_L
y_m = shots['Y']/100.0 * PITCH_W
shots['Distance'] = np.sqrt((x_m-PITCH_L)**2 + (y_m-GOAL_Y)**2)
dx = PITCH_L - x_m
shots['Angle'] = np.abs(np.degrees(np.arctan2(POST_R-y_m,dx) - np.arctan2(POST_L-y_m,dx)))

# Player rank (proxy: zero-fill so this notebook stays self-contained;
# the original processed file uses a richer playerank score, included here as fallback)
shots['Player_Rank'] = 0.0
# If processed file has Player_Rank, merge it in (keeps parity with original modelling)
proc = pd.read_csv(str(Paths.WYSCOUT_PROCESSED))
if 'Player_Rank' in proc.columns and len(proc)==len(shots):
    shots = shots.sort_index().reset_index(drop=True)
    proc  = proc.reset_index(drop=True)
    shots['Player_Rank'] = proc['Player_Rank'].values
    print('Merged Player_Rank from processed file by row order.')
else:
    print('Processed file row count differs -- using zero Player_Rank.')

FEATURES = ['X','Y','Distance','Angle','Left_Foot','Right_Foot','Header','First_Half','Player_Rank']
shots = shots[['matchId','Goal'] + FEATURES].dropna().reset_index(drop=True)
print(f'Shots after cleaning: {len(shots):,}')
print(f'matchId range: {shots["matchId"].min()} -> {shots["matchId"].max()}')

Raw shots: 8,451
Merged Player_Rank from processed file by row order.
Shots after cleaning: 8,451
matchId range: 2499719 -> 2500098


## Chronological split by matchId

In [5]:
# Sort by matchId (sequential by kickoff in this dataset)
shots = shots.sort_values('matchId').reset_index(drop=True)

# Find a matchId boundary that gives ~20% test rows
all_match_ids = shots['matchId'].drop_duplicates().sort_values().tolist()
split_match_idx = int(len(all_match_ids) * 0.8)
boundary_match_id = all_match_ids[split_match_idx]

train_mask = shots['matchId'] < boundary_match_id
X_train = shots.loc[train_mask, FEATURES]
y_train = shots.loc[train_mask, 'Goal']
X_test  = shots.loc[~train_mask, FEATURES]
y_test  = shots.loc[~train_mask, 'Goal']

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

neg, pos = (y_train==0).sum(), (y_train==1).sum()
scale_pos_weight = neg/pos

print(f'Train matches: {train_mask.groupby(shots.matchId).any().sum()}')
print(f'Test  matches: {(~train_mask).groupby(shots.matchId).any().sum()}')
print(f'Train shots: {len(X_train):,} | Test shots: {len(X_test):,}')
print(f'Train goal rate: {y_train.mean()*100:.2f}% | Test goal rate: {y_test.mean()*100:.2f}%')
print(f'scale_pos_weight: {scale_pos_weight:.2f}')

Train matches: 304
Test  matches: 76
Train shots: 6,818 | Test shots: 1,633
Train goal rate: 10.75% | Test goal rate: 11.08%
scale_pos_weight: 8.30


### Dummy

In [6]:
dummy = DummyClassifier(strategy='most_frequent', random_state=42)
dummy.fit(X_train_s, y_train)
y_prob_dummy = dummy.predict_proba(X_test_s)[:,1]
y_pred_dummy = dummy.predict(X_test_s)
acc_dummy = accuracy_score(y_test, y_pred_dummy)
auc_dummy = roc_auc_score(y_test, y_prob_dummy)
print(f'DUMMY -- Acc {acc_dummy*100:.2f}% | AUC {auc_dummy:.4f}')
print(classification_report(y_test, y_pred_dummy, target_names=['No Goal','Goal']))

DUMMY -- Acc 88.92% | AUC 0.5000
              precision    recall  f1-score   support

     No Goal       0.89      1.00      0.94      1452
        Goal       0.00      0.00      0.00       181

    accuracy                           0.89      1633
   macro avg       0.44      0.50      0.47      1633
weighted avg       0.79      0.89      0.84      1633



### Logistic Regression

In [7]:
lr = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)
lr.fit(X_train_s, y_train)
y_prob_lr = lr.predict_proba(X_test_s)[:,1]
y_pred_lr = lr.predict(X_test_s)
acc_lr = accuracy_score(y_test, y_pred_lr)
auc_lr = roc_auc_score(y_test, y_prob_lr)
print(f'LOGREG -- Acc {acc_lr*100:.2f}% | AUC {auc_lr:.4f}')
print(classification_report(y_test, y_pred_lr, target_names=['No Goal','Goal']))

LOGREG -- Acc 73.18% | AUC 0.8237
              precision    recall  f1-score   support

     No Goal       0.96      0.73      0.83      1452
        Goal       0.26      0.74      0.38       181

    accuracy                           0.73      1633
   macro avg       0.61      0.74      0.60      1633
weighted avg       0.88      0.73      0.78      1633



### Random Forest

In [8]:
rf = RandomForestClassifier(n_estimators=200, class_weight='balanced',
                             random_state=42, n_jobs=-1)
rf.fit(X_train_s, y_train)
y_prob_rf = rf.predict_proba(X_test_s)[:,1]
y_pred_rf = rf.predict(X_test_s)
acc_rf = accuracy_score(y_test, y_pred_rf)
auc_rf = roc_auc_score(y_test, y_prob_rf)
print(f'RF -- Acc {acc_rf*100:.2f}% | AUC {auc_rf:.4f}')
print(classification_report(y_test, y_pred_rf, target_names=['No Goal','Goal']))

RF -- Acc 89.04% | AUC 0.7940
              precision    recall  f1-score   support

     No Goal       0.91      0.97      0.94      1452
        Goal       0.51      0.24      0.33       181

    accuracy                           0.89      1633
   macro avg       0.71      0.61      0.63      1633
weighted avg       0.87      0.89      0.87      1633



### XGBoost

In [9]:
xgb = XGBClassifier(scale_pos_weight=scale_pos_weight, random_state=42,
                    eval_metric='auc', verbosity=0)
xgb.fit(X_train_s, y_train)
y_prob_xgb = xgb.predict_proba(X_test_s)[:,1]
y_pred_xgb = xgb.predict(X_test_s)
acc_xgb = accuracy_score(y_test, y_pred_xgb)
auc_xgb = roc_auc_score(y_test, y_prob_xgb)
print(f'XGB -- Acc {acc_xgb*100:.2f}% | AUC {auc_xgb:.4f}')
print(classification_report(y_test, y_pred_xgb, target_names=['No Goal','Goal']))

XGB -- Acc 81.57% | AUC 0.8034
              precision    recall  f1-score   support

     No Goal       0.94      0.85      0.89      1452
        Goal       0.31      0.55      0.40       181

    accuracy                           0.82      1633
   macro avg       0.62      0.70      0.64      1633
weighted avg       0.87      0.82      0.84      1633



### Cross-validation AUC (training partition only)

In [10]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

models_cv = [
    ('Dummy',               DummyClassifier(strategy='most_frequent', random_state=42)),
    ('Logistic Regression', LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)),
    ('Random Forest',       RandomForestClassifier(n_estimators=200, class_weight='balanced', random_state=42, n_jobs=-1)),
    ('XGBoost',             XGBClassifier(scale_pos_weight=scale_pos_weight, random_state=42, eval_metric='auc', verbosity=0)),
]
print('5-Fold CV AUC (training partition only):')
for name, model in models_cv:
    scores = cross_val_score(model, X_train_s, y_train, cv=cv, scoring='roc_auc', n_jobs=-1)
    print(f'  {name:<22} {scores.mean():.4f} +/- {scores.std():.4f}')

5-Fold CV AUC (training partition only):
  Dummy                  0.5000 +/- 0.0000
  Logistic Regression    0.7891 +/- 0.0183
  Random Forest          0.7646 +/- 0.0153
  XGBoost                0.7594 +/- 0.0085


## Side-by-side summary

Random-split AUCs (from `notebooks/cycle2_modelling.ipynb` summary): Dummy 0.5000, LR 0.7963, RF 0.7884, XGB 0.7871.

In [11]:
results = pd.DataFrame([
    {'Model':'Dummy',               'Chrono Acc %':acc_dummy*100,'Chrono AUC':auc_dummy,'Random AUC (orig)':0.500},
    {'Model':'Logistic Regression', 'Chrono Acc %':acc_lr*100,   'Chrono AUC':auc_lr,   'Random AUC (orig)':0.7963},
    {'Model':'Random Forest',       'Chrono Acc %':acc_rf*100,   'Chrono AUC':auc_rf,   'Random AUC (orig)':0.7884},
    {'Model':'XGBoost',             'Chrono Acc %':acc_xgb*100,  'Chrono AUC':auc_xgb,  'Random AUC (orig)':0.7871},
])
results['AUC Delta'] = (results['Chrono AUC']-results['Random AUC (orig)']).round(4)
results['Chrono Acc %'] = results['Chrono Acc %'].round(2)
results['Chrono AUC']   = results['Chrono AUC'].round(4)
print(results.to_string(index=False))

              Model  Chrono Acc %  Chrono AUC  Random AUC (orig)  AUC Delta
              Dummy         88.92      0.5000             0.5000     0.0000
Logistic Regression         73.18      0.8237             0.7963     0.0274
      Random Forest         89.04      0.7940             0.7884     0.0056
            XGBoost         81.57      0.8034             0.7871     0.0163
